# Stock Data Exploratory Analysis

> Module 1 - Markets, Data, and EDA

Transform raw stock prices into returns and descriptive statistics that support financial decision-making.

## Learning objectives
- Review missing values and usable date ranges across tickers.
- Compute simple returns and log returns.
- Summarize central tendency, dispersion, shape, percentiles, and correlations.
- Interpret descriptive statistics as financial evidence rather than isolated calculations.

## Lesson flow
1. Download adjusted close prices.
2. Audit missing values and coverage.
3. Build return series.
4. Compute summary statistics and correlations.

## Student deliverable
An EDA brief comparing at least three assets by return, volatility, skewness, kurtosis, and correlation.


## Setup

In [ ]:
import pandas as pd
import numpy as np
from datetime import date, timedelta

# Visualization 
import matplotlib.pyplot as plt
import seaborn as sns

# import pandas_datareader.data as web
import yfinance as yf

from scipy.stats import kurtosis


pd.set_option("display.max_columns",80)

In [ ]:
yesterday = str(date.today() - timedelta(days = 1))
print("Today's date:", yesterday)

## Data

In [ ]:
%%time
start_date = "2018-01-01"
tickers = ['MAT','DIS','KO', 'NVDA','PFE', "AAPL", "META", "TSLA", "^GSPC","MSFT", "NU"]
print(f"The number of stock to download are {len(tickers)}")
all_data = yf.download(tickers, start_date, yesterday)
all_data.info()

### Missing review

In [ ]:
df_missings_agg = all_data.isna().sum().sort_values()
df_missings_agg[
    df_missings_agg > 0
]

In [ ]:
adj_close_data_completed = all_data["Adj Close"].copy()
adj_close_data_completed.head()

In [ ]:
adj_close_data_completed[
    adj_close_data_completed.NU.notna()
].isna().sum()#.index.min()

In [ ]:
adj_close_data_completed[
    adj_close_data_completed.NU.notna()
].index.min()

In [ ]:
"AAPL" in adj_close_data_completed.columns

In [ ]:
"prueba" in adj_close_data_completed.columns

In [ ]:
adj_close_data = adj_close_data_completed.drop(columns=['NU','prueba'],errors='ignore').copy()
adj_close_data.head()

## Transformations

### Returns 
$$r_{t+1}=\frac{p_{t+1} - p_{t}}{p_{t}}$$

In [ ]:
(34.309586 - 38.105137) / 38.105137

In [ ]:
(35.774239- 34.309586) / 34.309586

In [ ]:
yield_data = adj_close_data.pct_change()#.dropna()
yield_data.head()

In [ ]:
yield_data = adj_close_data.pct_change().dropna()
yield_data.head()

### Log-returns

$$1+r_{t+1}= \frac{p_{t+1}}{p_{t}} = e^{log(\frac{p_{t+1}}{p_{t}} )}$$

In [ ]:
df_log = np.log(yield_data + 1)
df_log.head()

## Summary statistics

### Central tendency

#### Mean

In [ ]:
(
    df_log.pivot_table(
        index=df_log.index.year,
        aggfunc='mean'
    )*252
)

#### Median

The median is the value separating the higher half from the lower half of a data sample, a population, or a probability distribution.

$P(X \leq m) \geq \frac{1}{2} \: and \: P(X \geq m) \geq \frac{1}{2}$  




In [ ]:
(
    df_log.pivot_table(
        index=df_log.index.year,
        aggfunc='median'
    )*252
)

### Statistical dispersion

#### Standard deviation

In [ ]:
df_log.std().sort_values()

In [ ]:
df_log.pivot_table(
    index=df_log.index.year,
    aggfunc='std'
)

#### Variance

### Shape

#### Skewness
<img src="../../img/Relationship_between_mean_and_median_under_different_skewness.png" alt="MarineGEO circle logo" style="height: 350px; width:850px;"/>

In [ ]:
df_log.skew().sort_values()

In [ ]:
df_log.pivot_table(
    index=df_log.index.year,
    aggfunc='skew'
)

#### Kurtosis

<img src="../../img/Kurtosis.jpg" alt="MarineGEO circle logo" style="height: 350px; width:850px;"/>

In [ ]:
df_log.apply(kurtosis).sort_values()

In [ ]:
df_log.pivot_table(
    index=df_log.index.year,
    aggfunc='skew'
)

### Percentile

In [ ]:
df_log.quantile([0,.25,.5,.75,1])

### Correlation
<img src="../../img/1920px-Correlation_examples2.svg.png" alt="MarineGEO circle logo" style="height: 350px; width:850px;"/>

***Correlation** is a bivariate analysis that measures the strength of association between two variables and the direction of the relationship.  In terms of the strength of relationship, the value of the correlation coefficient varies between +1 and -1.  A value of ± 1 indicates a perfect degree of association between the two variables.  As the correlation coefficient value goes towards 0, the relationship between the two variables will be weaker.  The direction of the relationship is indicated by the sign of the coefficient; a + sign indicates a positive relationship and a – sign indicates a negative relationship.*

---

The three correlations I deal with are that one is a parametric method, Pearson correlation, and the others are a non-parametric method, Spearman and Kendall rank correlation.

#### Pearson correlation

$$
r = \frac{\sum(X - \overline{X})(Y - \overline{Y})}
{\sqrt{\sum(X-\overline{X})^{2}\cdot\sum(Y-\overline{Y})^{2}}}\\
~ \\
\begin{align}
    Where, ~ \overline{X} &= mean ~ of ~ X~variable\\
    \overline{Y} &= mean ~ of ~ Y ~ variable\\
\end{align}
$$

Assumptions:

- Each observation should have a pair of values.

- Each variable should be continuous.

- Each variable should be normally distributed.

- It should be an absence of outliers.

- It assumes linearity and homoscedasticity.

In [ ]:
df_log.corr(method = 'pearson')

#### Spearman rank correlation

$$
\rho = \frac{\sum_{i=1}^{n}(R(x_i) - \overline{R(x)})(R(y_i) - \overline{R(y)})}
{\sqrt{\sum_{i=1}^{n}(R(x_i) - \overline{R(x)})^{2}\cdot\sum_{i=1}^{n}(R(y_i)-\overline{R(y)})^{2}}}
= 1 - \frac{6\sum_{i=1}^{n}(R(x_i) - R(y_i))^{2}}{n(n^{2} - 1)}\\
~ \\
\begin{align}
    Where, ~ R(x_i) &= rank ~ of ~ x_i\\
    R(y_i) &= rank ~ of ~ y_i\\
    \overline{R(x)} &=mean ~ rank ~ of ~ x\\
    \overline{R(y)} &=mean ~ rank ~ of ~ y\\
    n &= number ~ of ~ pairs
\end{align}
$$

Assumptions:

- Pairs of observations are independent.

- Two variables should be measured on an ordinal, interval or ratio scale.

- It assumes that there is a monotonic relationship between the two variables.

In [ ]:
df_log.corr(method = 'spearman')

#### Kendall rank correlation
$$
\tau = \frac{n_c - n_d}{n_c + n_d} = \frac{n_c - n_d}{n(n-1)/2}\\ 
~ \\
\begin{align}
    Where, ~ n_c &= number ~ of ~ concordant ~ pairs\\
    n_d &= number ~ of ~ discordant ~ pairs\\
    n &= number ~ of ~ pairs
\end{align}
$$
Assumptions:
- It's the same as assumptions of Spearman rank correlation.

In [ ]:
df_log.corr(method = 'kendall')

#### Comparison of Each Correlation
**Pearson correlation vs Spearman and Kendall correlations**

- Non-parametric correlations are less powerful because they use less information in their calculations. In the case of Pearson correlation uses information about the mean and deviation from the mean, while non-parametric correlations use only the ordinal information and scores of pairs.

- In the case of non-parametric correlation, it's possible that the X and Y values can be continuous or ordinal, and approximate normal distributions for X and Y are not required. But in the case of Pearson correlation, it assumes the distributions of X and Y should be normal distribution and also be continuous.

- Correlation coefficients only measure linear (Pearson) or monotonic (Spearman and Kendall) relationships.

**Spearman correlation vs Kendall correlation**

- In the normal case, Kendall correlation is more robust and efficient than Spearman correlation. It means that Kendall correlation is preferred when there are small samples or some outliers.

- Kendall correlation has an O(n^2) computation complexity comparing with O(n logn) of Spearman correlation, where n is the sample size.

- Spearman’s rho usually is larger than Kendall’s tau.

- The interpretation of Kendall’s tau in terms of the probabilities of observing the agreeable (concordant) and non-agreeable (discordant) pairs is very direct.

### Covariance

In [ ]:
df_log.cov()

## Plots

### Histograms

In [ ]:
adj_close_data.hist(figsize=(12,10), bins=50);

In [ ]:
df_log.hist(figsize=(12,10), bins=50);

### Barplots

In [ ]:
df_log.plot();

In [ ]:
df_log.plot(
    figsize=(18,10),
    alpha=0.5
);

In [ ]:
min_scale=df_log.min().min()
max_scale=df_log.max().max()
print(f"The global minimum and maximum {(min_scale, max_scale)}")
# print(f"The global minimum: {min_scale}")
for col in df_log:
    print(f"\n")
    df_log[col].plot(figsize=(15,7))
    plt.title(f"Barplot of daily log-returns for stock: {col}")
    plt.ylim(min_scale*1.05, max_scale*1.05)
    plt.axhline(y = min_scale, color='r', linestyle='--')
    plt.axhline(y = max_scale, color='r', linestyle='--')
    plt.axhline(y = -0.1, color='gold', linestyle='--')
    plt.axhline(y = 0, color='green', linestyle='--')
    plt.show();

In [ ]:
df_log.std().sort_values().plot(kind='barh');

In [ ]:
df_log.pivot_table(
    index=df_log.index.year,
    aggfunc='std'
).plot(
    figsize=(14,8)
);

### Lineplot

In [ ]:
adj_close_data_completed[
    adj_close_data_completed.NU.notna()
].plot(
    figsize=(12,8)
);

In [ ]:
adj_close_data_completed[
    adj_close_data_completed.NU.notna()
].drop(
    columns=["^GSPC"]
).plot(
    figsize=(12,8)
);

In [ ]:
for col in adj_close_data:
    print(f"\n")
    adj_close_data[col].plot(figsize=(15,7))
    plt.title(f"Barplot of daily adj-close price for stock: {col}")
    plt.show();

In [ ]:
(
    df_log.pivot_table(
        index=df_log.index.year,
        aggfunc='mean'
    )*252
).plot(
    figsize=(14,8)
);
plt.axhline(y=0, color='y', linestyle='--');

### Boxplot

In [ ]:
df_log.boxplot(figsize=(14, 10), grid=False)
plt.title("Daily log-returns of the stocks");

### Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(14,7))    
sns.heatmap(
    df_log.corr(method='pearson'), 
    vmin=-1, 
    vmax=1,
    annot=True,cmap="rocket_r",
    ax=ax
)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14,7))    
sns.heatmap(
    df_log.corr(method='spearman'), 
    vmin=-1, 
    vmax=1,
    annot=True,cmap="rocket_r",
    ax=ax
)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14,7))    
sns.heatmap(
    df_log.corr(method='kendall'), 
    vmin=-1, 
    vmax=1,
    annot=True,cmap="rocket_r",
    ax=ax
)
plt.show()

## References
- https://www.tylervigen.com/spurious-correlations
- https://www.kaggle.com/code/kiyoung1027/correlation-pearson-spearman-and-kendall?scriptVersionId=25999032#679923